In [ ]:
import glob, json, subprocess, sys, torch, zipfile
assert torch.cuda.is_available()
name = torch.cuda.get_device_name(0)
assert '6000' in name, name
print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version',
                     '--format=csv'], capture_output=True, text=True, check=True).stdout)
modules = glob.glob('/kaggle/input/**/faiss/__init__.py', recursive=True)
if modules:
    assert len(modules) == 1, modules
    sys.path.insert(0, str(__import__('pathlib').Path(modules[0]).parent.parent))
else:
    archives = glob.glob('/kaggle/input/**/faiss_runtime.zip', recursive=True)
    assert len(archives) == 1, archives
    runtime = '/kaggle/working/faiss_runtime_smoke'
    with zipfile.ZipFile(archives[0]) as z: z.extractall(runtime)
    sys.path.insert(0, runtime)
import faiss
result = {'gpu': name, 'vram': torch.cuda.get_device_properties(0).total_memory,
          'faiss': faiss.__version__, 'faiss_gpus': faiss.get_num_gpus()}
print(json.dumps(result))
open('/kaggle/working/faiss_runtime_smoke.json', 'w').write(json.dumps(
    {'gpu': name, 'faiss': faiss.__version__, 'faiss_gpus': faiss.get_num_gpus()}))
